# AI-Powered Project Planning & Risk Forecasting App

This notebook walks through the full pipeline of the application step by step:

1. Environment setup & configuration
2. Task schema definition
3. AI task generation via Groq
4. Graph construction & critical path
5. Monte Carlo simulation
6. Risk metrics (P50, P80, delay probability)
7. Risk driver ranking
8. Scenario comparison
9. Visualization

> **Tip:** Set `APP_MODE=mock` in your `.env` to run the full notebook without consuming Groq API tokens.

---
## 1. Install & Import Dependencies

In [1]:
# Install dependencies if not already installed
!pip install groq networkx numpy pandas plotly nbformat python-dotenv pydantic streamlit --quiet

In [2]:
import os
import json
import copy
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pydantic import BaseModel, Field, field_validator
from typing import List
from dotenv import load_dotenv
import plotly.express as px

warnings.filterwarnings('ignore')
load_dotenv()

print("All dependencies imported successfully.")

All dependencies imported successfully.


---
## 2. Configuration

In [3]:
# Configuration
APP_MODE        = os.getenv("APP_MODE", "mock").strip().lower()  # "real" or "mock"
GROQ_API_KEY    = os.getenv("GROQ_API_KEY", "")
GROQ_MODEL      = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
DEFAULT_ITERS   = 1000
MIN_DURATION    = 0.1

print(f"APP_MODE  : {APP_MODE}")
print(f"GROQ_MODEL: {GROQ_MODEL}")
print(f"API KEY   : {'set' if GROQ_API_KEY else 'missing (mock mode will still work)'}")

APP_MODE  : mock
GROQ_MODEL: llama-3.3-70b-versatile
API KEY   : missing (mock mode will still work)


---
## 3. Task Schema (Pydantic)

Every task generated by the LLM is validated against this schema before entering the pipeline. This prevents malformed data from reaching the graph or simulation engines.

In [4]:
class Task(BaseModel):
    id: str                  = Field(..., description="Unique task ID like T1, T2...")
    name: str                = Field(..., min_length=2)
    mean_duration: float     = Field(..., gt=0, description="Expected duration in days")
    std_dev: float           = Field(..., ge=0, description="Standard deviation in days")
    dependencies: List[str]  = Field(default_factory=list)
    risk_factor: float       = Field(..., ge=0, le=1)

    @field_validator("id")
    @classmethod
    def id_format(cls, v: str) -> str:
        v = v.strip()
        if not v.startswith("T"):
            raise ValueError("Task id must start with 'T' (e.g., T1).")
        return v

    @field_validator("name")
    @classmethod
    def clean_name(cls, v: str) -> str:
        return v.strip()


class TaskPlan(BaseModel):
    tasks: List[Task]

    @field_validator("tasks")
    @classmethod
    def unique_ids(cls, tasks: List[Task]) -> List[Task]:
        ids = [t.id for t in tasks]
        if len(ids) != len(set(ids)):
            raise ValueError("Duplicate task ids found.")
        return tasks


print("Task schema defined.")

# Quick validation test
test_task = Task(id="T1", name="Requirements", mean_duration=8.0, std_dev=2.0, dependencies=[], risk_factor=0.3)
print(f"   Sample task: {test_task.id} — {test_task.name} ({test_task.mean_duration} days)")

Task schema defined.
   Sample task: T1 — Requirements (8.0 days)


---
## 4. Mock Data

Used when `APP_MODE=mock` to allow full pipeline exploration without API calls.

In [5]:
def mock_task_plan() -> dict:
    return {
        "tasks": [
            {"id": "T1", "name": "Requirements & Scope",       "mean_duration": 8,  "std_dev": 2,   "dependencies": [],           "risk_factor": 0.30},
            {"id": "T2", "name": "Project Plan & Milestones",  "mean_duration": 6,  "std_dev": 1.5, "dependencies": ["T1"],        "risk_factor": 0.20},
            {"id": "T3", "name": "Design & Architecture",      "mean_duration": 10, "std_dev": 3,   "dependencies": ["T1"],        "risk_factor": 0.35},
            {"id": "T4", "name": "Implementation",             "mean_duration": 18, "std_dev": 6,   "dependencies": ["T3"],        "risk_factor": 0.45},
            {"id": "T5", "name": "Integration",                "mean_duration": 10, "std_dev": 4,   "dependencies": ["T4"],        "risk_factor": 0.50},
            {"id": "T6", "name": "Testing & QA",               "mean_duration": 12, "std_dev": 4,   "dependencies": ["T5"],        "risk_factor": 0.55},
            {"id": "T7", "name": "Documentation & Training",   "mean_duration": 6,  "std_dev": 2,   "dependencies": ["T4"],        "risk_factor": 0.25},
            {"id": "T8", "name": "Release & Handover",         "mean_duration": 4,  "std_dev": 1,   "dependencies": ["T6", "T7"],  "risk_factor": 0.20},
        ]
    }

print("Mock data ready.")

Mock data ready.


---
## 5. AI Task Generation via Groq

The LLM receives a strict prompt requiring JSON-only output. The response is validated by the Pydantic schema before proceeding.

In [6]:
def build_prompt(project_description: str, max_tasks: int = 12) -> str:
    return f"""
You are a senior project planning assistant.

Goal: Generate a project task plan with dependencies and uncertainty estimates suitable for Monte Carlo simulation.

Hard requirements:
- Output ONLY valid JSON. No markdown, no commentary.
- JSON must match this shape exactly:
  {{
    "tasks": [
      {{
        "id": "T1",
        "name": "Short Task Name",
        "mean_duration": 10,
        "std_dev": 2,
        "dependencies": [],
        "risk_factor": 0.2
      }}
    ]
  }}

Rules:
- Create between 8 and {max_tasks} tasks.
- Task IDs must be sequential starting from T1, T2, T3... (no gaps).
- Dependencies must reference earlier IDs only (no cycles).
- mean_duration is in days (>= 1).
- std_dev should be 10%–40% of mean_duration.
- risk_factor is 0..1.

Project description:
{project_description.strip()}
""".strip()


def _safe_json_loads(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").replace("json", "", 1).strip()
    return json.loads(text)


def generate_task_plan(project_description: str, max_tasks: int = 12) -> TaskPlan:
    if APP_MODE == "mock":
        print("   [mock mode] Returning built-in sample plan.")
        return TaskPlan.model_validate(mock_task_plan())

    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY)
    prompt = build_prompt(project_description, max_tasks=max_tasks)

    resp = client.chat.completions.create(
        model=GROQ_MODEL,
        temperature=0.25,
        messages=[{"role": "user", "content": prompt}],
    )
    content = resp.choices[0].message.content or ""
    data = _safe_json_loads(content)
    return TaskPlan.model_validate(data)


print("Task generation functions defined.")

Task generation functions defined.


In [7]:
# ─── Run Task Generation ──────────────────────────────────────────────────────
PROJECT_DESCRIPTION = "Launch a customer portal with authentication, billing, and analytics"
DEADLINE_DAYS       = 90.0

print(f"Project : {PROJECT_DESCRIPTION}")
print(f"Deadline: {DEADLINE_DAYS} days")
print()

plan  = generate_task_plan(PROJECT_DESCRIPTION)
tasks = [t.model_dump() for t in plan.tasks]

df_tasks = pd.DataFrame(tasks).rename(columns={
    "mean_duration" : "Expected Days",
    "std_dev"       : "Uncertainty (Std Dev)",
    "risk_factor"   : "Risk Factor",
    "dependencies"  : "Dependencies"
})

print(f"Generated {len(tasks)} tasks")
df_tasks

Project : Launch a customer portal with authentication, billing, and analytics
Deadline: 90.0 days

   [mock mode] Returning built-in sample plan.
Generated 8 tasks


,id,name,Expected Days,Uncertainty (Std Dev),Dependencies,Risk Factor
0,T1,Requirements & Scope,8.0,2.0,[],0.30
1,T2,Project Plan & Milestones,6.0,1.5,[T1],0.20
2,T3,Design & Architecture,10.0,3.0,[T1],0.35
3,T4,Implementation,18.0,6.0,[T3],0.45
4,T5,Integration,10.0,4.0,[T4],0.50
5,T6,Testing & QA,12.0,4.0,[T5],0.55
6,T7,Documentation & Training,6.0,2.0,[T4],0.25
7,T8,Release & Handover,4.0,1.0,"[T6, T7]",0.20


---
## 6. Graph Construction & Critical Path

Tasks are assembled into a **Directed Acyclic Graph (DAG)**. The critical path is the longest chain of dependent tasks by mean duration — it determines the minimum possible project completion time.

In [8]:
def build_project_graph(tasks: list[dict]) -> nx.DiGraph:
    G = nx.DiGraph()
    for t in tasks:
        G.add_node(t["id"], **t)
    for t in tasks:
        for dep in t.get("dependencies", []):
            if dep not in G.nodes:
                raise ValueError(f"Dependency {dep} not found for task {t['id']}")
            G.add_edge(dep, t["id"])
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError("Project graph contains cycles — not a valid DAG.")
    return G


def critical_path_by_mean(G: nx.DiGraph) -> tuple[list[str], float]:
    path  = nx.dag_longest_path(G)
    total = sum(float(G.nodes[n]["mean_duration"]) for n in path)
    return path, total


G  = build_project_graph(tasks)
cp, cp_days = critical_path_by_mean(G)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"\nCritical Path: {' → '.join(cp)}")
print(f"Baseline duration (mean): {cp_days:.1f} days")

Graph built: 8 nodes, 8 edges

Critical Path: T1 → T3 → T4 → T5 → T6 → T8
Baseline duration (mean): 62.0 days


In [9]:
# Visualize the Dependency Graph
def plot_dependency_graph(G: nx.DiGraph, critical_path: list[str]):
    pos = nx.spring_layout(G, seed=42, k=2)
    cp_set = set(critical_path)

    edge_x, edge_y = [], []
    for u, v in G.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    node_x   = [pos[n][0] for n in G.nodes()]
    node_y   = [pos[n][1] for n in G.nodes()]
    labels   = [G.nodes[n]["name"] for n in G.nodes()]
    colors   = ["#EF553B" if n in cp_set else "#636EFA" for n in G.nodes()]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                             line=dict(width=1.5, color="#aaa"), hoverinfo="none"))
    fig.add_trace(go.Scatter(x=node_x, y=node_y, mode="markers+text",
                             marker=dict(size=28, color=colors, line=dict(width=2, color="white")),
                             text=[n for n in G.nodes()],
                             customdata=labels,
                             hovertemplate="<b>%{customdata}</b><extra></extra>",
                             textposition="middle center",
                             textfont=dict(color="white", size=11)))
    fig.update_layout(
        title="Task Dependency Graph  <span style='color:#EF553B'>■</span> Critical Path",
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=480,
        plot_bgcolor="#0e1117",
        paper_bgcolor="#0e1117",
        font=dict(color="white")
    )
    try:
        fig.show()
    except ValueError as e:
        if "nbformat>=4.2.0" in str(e):
            fig.show(renderer="browser")
        else:
            raise

plot_dependency_graph(G, cp)

---
## 7. Monte Carlo Simulation

For each iteration, every task's duration is sampled from a Normal distribution using its `mean_duration` and `std_dev`. The project completion time is the sum of sampled durations along the critical path. Running thousands of iterations produces a distribution of possible outcomes.

In [10]:
def run_monte_carlo(G: nx.DiGraph, iterations: int = 1000, seed: int | None = None) -> np.ndarray:
    rng     = np.random.default_rng(seed)
    results = np.zeros(iterations, dtype=float)
    nodes   = list(G.nodes)
    path    = nx.dag_longest_path(G)  # structural critical path

    for i in range(iterations):
        sampled = {}
        for n in nodes:
            mean = float(G.nodes[n]["mean_duration"])
            std  = float(G.nodes[n]["std_dev"])
            d    = rng.normal(mean, std) if std > 0 else mean
            sampled[n] = max(float(d), MIN_DURATION)
        results[i] = sum(sampled[n] for n in path)

    return results


N_ITERATIONS = 1000
completion   = run_monte_carlo(G, iterations=N_ITERATIONS, seed=7)

print(f"Simulation complete: {N_ITERATIONS:,} iterations")
print(f"   Min: {completion.min():.1f} days")
print(f"   Max: {completion.max():.1f} days")
print(f"   Mean: {completion.mean():.1f} days")

Simulation complete: 1,000 iterations
   Min: 34.8 days
   Max: 89.8 days
   Mean: 61.4 days


---
## 8. Risk Metrics

- **P50** — 50% of simulations finish by this date (median)
- **P80** — 80% of simulations finish by this date (recommended commitment date)
- **Delay Probability** — % of simulations that exceed the deadline

In [11]:
def compute_metrics(completion_times: np.ndarray, deadline: float) -> dict:
    return {
        "mean"             : float(np.mean(completion_times)),
        "p50"              : float(np.percentile(completion_times, 50)),
        "p80"              : float(np.percentile(completion_times, 80)),
        "delay_probability": float(np.mean(completion_times > deadline)),
    }


m = compute_metrics(completion, DEADLINE_DAYS)

print("Risk Metrics")
print(f"   Mean completion  : {m['mean']:.1f} days")
print(f"   P50              : {m['p50']:.1f} days")
print(f"   P80              : {m['p80']:.1f} days")
print(f"   Delay probability: {m['delay_probability']*100:.1f}%  (vs {DEADLINE_DAYS}-day deadline)")

Risk Metrics
   Mean completion  : 61.4 days
   P50              : 61.5 days
   P80              : 69.1 days
   Delay probability: 0.0%  (vs 90.0-day deadline)


In [12]:
# Executive Summary
def executive_summary(delay_prob: float, p80: float, top_driver: str | None) -> str:
    risk = "LOW" if delay_prob < 0.25 else ("MEDIUM" if delay_prob < 0.55 else "HIGH")
    driver_text = f" Primary delay driver: {top_driver}." if top_driver else ""
    return (
        f"Based on {N_ITERATIONS:,} simulations, there is a {delay_prob*100:.1f}% probability of "
        f"missing the {int(DEADLINE_DAYS)}-day deadline (risk level: {risk}). "
        f"Recommended commitment date: {p80:.1f} days (P80).{driver_text}"
    )

print("EXECUTIVE SUMMARY")
print("-" * 60)

EXECUTIVE SUMMARY
------------------------------------------------------------


---
## 9. Monte Carlo Distribution Chart

In [13]:
def plot_completion_histogram(completion_times, deadline, p50, p80):
    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=completion_times, nbinsx=40,
        name="Simulated Completion",
        marker_color="#636EFA", opacity=0.75
    ))
    fig.add_vline(x=deadline, line_dash="dash",  line_color="#EF553B", line_width=2,
                  annotation_text=f"Deadline ({deadline:.0f}d)",
                  annotation_font_color="#EF553B")
    fig.add_vline(x=p50,      line_dash="dot",   line_color="#00CC96", line_width=2,
                  annotation_text=f"P50 ({p50:.1f}d)",
                  annotation_font_color="#00CC96")
    fig.add_vline(x=p80,      line_dash="dot",   line_color="#FECB52", line_width=2,
                  annotation_text=f"P80 ({p80:.1f}d)",
                  annotation_font_color="#FECB52")

    fig.update_layout(
        title="Monte Carlo Completion Distribution",
        xaxis_title="Completion Time (days)",
        yaxis_title="Frequency",
        bargap=0.05,
        height=420,
        plot_bgcolor="#0e1117",
        paper_bgcolor="#0e1117",
        font=dict(color="white")
    )
    try:
        fig.show()
    except ValueError as e:
        if "nbformat>=4.2.0" in str(e):
            fig.show(renderer="browser")
        else:
            raise

plot_completion_histogram(completion, DEADLINE_DAYS, m["p50"], m["p80"])

---
## 10. Risk Driver Ranking

Across simulations, we count how frequently each task lands on the critical path. Tasks with the highest frequency are the **primary delay drivers** — reducing their uncertainty has the greatest impact on overall schedule risk.

In [14]:
import streamlit as st
import plotly.graph_objects as go

def plot_completion_histogram(completion_times, deadline, p50, p80):
    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=completion_times,
        nbinsx=40,
        name="Simulated Completion",
        marker_color="#636EFA",
        opacity=0.75
    ))

    fig.add_vline(x=deadline, line_dash="dash", line_color="#EF553B", line_width=2,
                  annotation_text=f"Deadline ({deadline:.0f}d)",
                  annotation_font_color="#EF553B")

    fig.add_vline(x=p50, line_dash="dot", line_color="#00CC96", line_width=2,
                  annotation_text=f"P50 ({p50:.1f}d)",
                  annotation_font_color="#00CC96")

    fig.add_vline(x=p80, line_dash="dot", line_color="#FECB52", line_width=2,
                  annotation_text=f"P80 ({p80:.1f}d)",
                  annotation_font_color="#FECB52")

    fig.update_layout(
        title="Monte Carlo Completion Distribution",
        xaxis_title="Completion Time (days)",
        yaxis_title="Frequency",
        bargap=0.05,
        height=420,
        plot_bgcolor="#0e1117",
        paper_bgcolor="#0e1117",
        font=dict(color="white")
    )

    st.plotly_chart(fig, use_container_width=True)

In [15]:
# Risk Drivers Bar Chart
if "df_drivers" not in globals() or df_drivers is None or len(df_drivers) == 0:
    if "top_drivers" in globals() and top_drivers:
        df_drivers = pd.DataFrame(top_drivers)
    elif "rank_delay_drivers" in globals() and "G" in globals():
        top_drivers = rank_delay_drivers(G, iterations=500, seed=42)
        df_drivers = pd.DataFrame(top_drivers)
    elif "G" in globals() and "cp" in globals():
        # Fallback ranking from the current critical path if simulation ranking is unavailable.
        df_drivers = pd.DataFrame([
            {
                "Task Name": G.nodes[n].get("name", n),
                "CP Freq (%)": 100.0 if n in set(cp) else 0.0,
            }
            for n in G.nodes()
        ]).sort_values("CP Freq (%)", ascending=False).head(10)
    else:
        raise ValueError("df_drivers is undefined and no inputs are available to build it.")

fig = px.bar(
    df_drivers,
    x="CP Freq (%)", y="Task Name",
    orientation="h",
    color="CP Freq (%)",
    color_continuous_scale="Reds",
    title="Risk Driver Ranking - Critical Path Frequency (%)",
    labels={"CP Freq (%)": "Critical Path Frequency (%)", "Task Name": ""}
)
fig.update_layout(
    height=400,
    yaxis=dict(autorange="reversed"),
    plot_bgcolor="#0e1117",
    paper_bgcolor="#0e1117",
    font=dict(color="white"),
    coloraxis_showscale=False
)
try:
    fig.show()
except ValueError as e:
    if "nbformat>=4.2.0" in str(e):
        fig.show(renderer="browser")
    else:
        raise


---
## 11. Scenario Comparison

Three scenarios are compared side by side:

| Scenario | Description |
|---|---|
| **Baseline** | As generated by the LLM |
| **Aggressive Deadline** | Same plan evaluated against a −15% shorter deadline |
| **Increased Capacity** | Mean durations and std deviations reduced by 15% (team works faster) |

In [16]:
def apply_capacity_boost(tasks: list[dict], factor: float = 0.85) -> list[dict]:
    boosted = copy.deepcopy(tasks)
    for t in boosted:
        t["mean_duration"] = max(1.0, float(t["mean_duration"]) * factor)
        t["std_dev"]       = max(0.0, float(t["std_dev"]) * factor)
    return boosted


# Scenario 1 — Baseline (already computed)
tight_deadline   = DEADLINE_DAYS * 0.85
delay_tight      = float((completion > tight_deadline).mean())

# Scenario 2 — Increased Capacity
boosted_tasks    = apply_capacity_boost(tasks, factor=0.85)
G_boost          = build_project_graph(boosted_tasks)
completion_boost = run_monte_carlo(G_boost, iterations=N_ITERATIONS, seed=11)
m_boost          = compute_metrics(completion_boost, DEADLINE_DAYS)

df_scenarios = pd.DataFrame([
    {
        "Scenario"       : "Baseline",
        "Delay Prob (%)" : round(m["delay_probability"] * 100, 1),
        "P80 (days)"     : round(m["p80"], 1),
        "Mean (days)"    : round(m["mean"], 1),
        "Notes"          : f"Deadline: {DEADLINE_DAYS:.0f} days"
    },
    {
        "Scenario"       : "Aggressive Deadline (−15%)",
        "Delay Prob (%)" : round(delay_tight * 100, 1),
        "P80 (days)"     : round(m["p80"], 1),
        "Mean (days)"    : round(m["mean"], 1),
        "Notes"          : f"Deadline eval: {tight_deadline:.1f} days"
    },
    {
        "Scenario"       : "Increased Capacity (+15% faster)",
        "Delay Prob (%)" : round(m_boost["delay_probability"] * 100, 1),
        "P80 (days)"     : round(m_boost["p80"], 1),
        "Mean (days)"    : round(m_boost["mean"], 1),
        "Notes"          : "Means & std dev reduced by 15%"
    },
])

print("Scenario Comparison")
df_scenarios

Scenario Comparison


,Scenario,Delay Prob (%),P80 (days),Mean (days),Notes
0,Baseline,0.0,69.1,61.4,Deadline: 90 days
1,Aggressive Deadline (−15%),5.2,69.1,61.4,Deadline eval: 76.5 days
2,Increased Capacity (+15% faster),0.0,59.9,53.2,Means & std dev reduced by 15%


In [17]:
# Scenario Comparison Chart
fig = go.Figure()

colors = ["#636EFA", "#EF553B", "#00CC96"]
for i, row in df_scenarios.iterrows():
    fig.add_trace(go.Bar(
        name=row["Scenario"],
        x=["Delay Prob (%)", "P80 (days)", "Mean (days)"],
        y=[row["Delay Prob (%)"], row["P80 (days)"], row["Mean (days)"]],
        marker_color=colors[i]
    ))

fig.update_layout(
    barmode="group",
    title="Scenario Comparison",
    height=420,
    plot_bgcolor="#0e1117",
    paper_bgcolor="#0e1117",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(0,0,0,0)")
)
try:
    fig.show()
except ValueError as e:
    if "nbformat>=4.2.0" in str(e):
        fig.show(renderer="browser")
    else:
        raise

---
## 12. Full Executive Summary

In [18]:
top_driver_name = None
if "df_drivers" in globals() and df_drivers is not None and len(df_drivers) > 0:
    top_driver_name = str(df_drivers.iloc[0]["Task Name"])
elif "top_drivers" in globals() and top_drivers:
    top_driver_name = str(top_drivers[0].get("Task Name", "")) or None

summary = executive_summary(m["delay_probability"], m["p80"], top_driver_name)

print("=" * 65)
print("  EXECUTIVE SUMMARY")
print("=" * 65)
print()
print(f"  Project  : {PROJECT_DESCRIPTION}")
print(f"  Deadline : {DEADLINE_DAYS:.0f} days")
print()
print(f"  {summary}")
print()
print("  Key Metrics:")
print(f"    Mean completion   : {m['mean']:.1f} days")
print(f"    P50               : {m['p50']:.1f} days")
print(f"    P80 (commit date) : {m['p80']:.1f} days")
print(f"    Delay probability : {m['delay_probability']*100:.1f}%")
print(f"    Top delay driver  : {top_driver_name if top_driver_name else 'N/A'}")
print("=" * 65)


  EXECUTIVE SUMMARY

  Project  : Launch a customer portal with authentication, billing, and analytics
  Deadline : 90 days

  Based on 1,000 simulations, there is a 0.0% probability of missing the 90-day deadline (risk level: LOW). Recommended commitment date: 69.1 days (P80). Primary delay driver: Requirements & Scope.

  Key Metrics:
    Mean completion   : 61.4 days
    P50               : 61.5 days
    P80 (commit date) : 69.1 days
    Delay probability : 0.0%
    Top delay driver  : Requirements & Scope


---
## 13. Next Steps

| Step | Action |
|---|---|
| **Run the full app** | `streamlit run app.py` |
| **Switch to real mode** | Set `APP_MODE=real` in `.env` and add your `GROQ_API_KEY` |
| **Test other projects** | Change `PROJECT_DESCRIPTION` in cell 5 and re-run all |
| **Tune simulation** | Increase `N_ITERATIONS` to 5,000 for smoother distributions |
| **Deploy** | Push to GitHub → Streamlit Community Cloud or Hugging Face Spaces |